In [0]:
from pyspark.sql import functions as F

# O caminho do ficheiro no Volume 
caminho_ficheiro = '/Volumes/workspace/default/dados_brutos/dia46_bronze_eventos_ga4.csv'

# 1. Ler o CSV com PySpark
df = spark.read.csv(caminho_ficheiro, header=True, inferSchema=True)

# 2. Aplicar as transformações da camada Silver
silver = (df
  .withColumn('event_date', F.to_date(F.col('event_date').cast('string'), 'yyyyMMdd'))
  .fillna({'country': 'NÃO INFORMADO'})
  .withColumn('country', F.upper('country')))

# 3. Mostrar as primeiras 5 linhas e a estrutura da tabela
silver.show(5)
silver.printSchema()

+----------+----------------+---------------+--------------------+-------------+--------------------+--------------------+
|event_date| event_timestamp|     event_name|      user_pseudo_id|      country|_ingestion_timestamp|        _source_file|
+----------+----------------+---------------+--------------------+-------------+--------------------+--------------------+
|2026-09-14|1789419999785683|user_engagement|ZpAtbc29eo4NEk8M8...|UNITED STATES|2026-09-18 16:09:...|dia46_bronze_even...|
|2026-09-14|1789419999020279|user_engagement|ZpAtbc29eo4NEk8M8...|UNITED STATES|2026-09-18 16:09:...|dia46_bronze_even...|
|2026-09-14|1789419999019606|      page_view|ZpAtbc29eo4NEk8M8...|UNITED STATES|2026-09-18 16:09:...|dia46_bronze_even...|
|2026-09-14|1789419659679233|user_engagement|ZpAtbc29eo4NEk8M8...|UNITED STATES|2026-09-18 16:09:...|dia46_bronze_even...|
|2026-09-14|1789419652022740|         scroll|ZpAtbc29eo4NEk8M8...|UNITED STATES|2026-09-18 16:09:...|dia46_bronze_even...|
+----------+----

In [0]:
#  Gravar como tabela Delta
silver.write.format('delta').mode('overwrite').saveAsTable('silver_eventos_ga4')

#  Observar o plano de execução (Lazy Evaluation)
print("\n--- Plano de Execução (Lazy Evaluation) ---")
silver.explain()


--- Plano de Execução (Lazy Evaluation) ---
== Physical Plan ==
PhotonResultStage
+- PhotonColumnarToRow
   +- PhotonProject [cast(gettimestamp(cast(event_date#10099 as string), yyyyMMdd, TimestampType, try_to_date, Some(Etc/UTC), true) as date) AS event_date#10146, event_timestamp#10100L, event_name#10101, user_pseudo_id#10102, upper(coalesce(country#10103, NÃO INFORMADO)) AS country#10150, _ingestion_timestamp#10104, _source_file#10105]
      +- PhotonRowToColumnar
         +- FileScan csv [event_date#10099,event_timestamp#10100L,event_name#10101,user_pseudo_id#10102,country#10103,_ingestion_timestamp#10104,_source_file#10105] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[dbfs:/Volumes/workspace/default/dados_brutos/dia46_bronze_eventos_ga4...., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<event_date:int,event_timestamp:bigint,event_name:string,user_pseudo_id:string,country:stri...


== Photon Explanation ==
The query is fully supp

In [0]:
%sql
SELECT country, COUNT(DISTINCT user_pseudo_id) AS usuarios
FROM silver_eventos_ga4
GROUP BY country;

country,usuarios
UNITED STATES,1
